In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

from skopt import BayesSearchCV
from skopt.space import Real, Integer

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import warnings, os, joblib
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../dataset/Clean_CICDDOS_Final_Dataset.csv")

print("Train Shape:", df.shape)
# Train Shape: (461158, 67)

Train Shape: (346873, 67)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346873 entries, 0 to 346872
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Protocol                     346873 non-null  int64  
 1   Flow_Duration                346873 non-null  int64  
 2   Total_Fwd_Packets            346873 non-null  int64  
 3   Total_Backward_Packets       346873 non-null  int64  
 4   Total_Length_of_Fwd_Packets  346873 non-null  float64
 5   Total_Length_of_Bwd_Packets  346873 non-null  float64
 6   Fwd_Packet_Length_Max        346873 non-null  float64
 7   Fwd_Packet_Length_Min        346873 non-null  float64
 8   Fwd_Packet_Length_Mean       346873 non-null  float64
 9   Fwd_Packet_Length_Std        346873 non-null  float64
 10  Bwd_Packet_Length_Max        346873 non-null  float64
 11  Bwd_Packet_Length_Min        346873 non-null  float64
 12  Bwd_Packet_Length_Mean       346873 non-null  float64
 13 

In [ ]:
# X = df.drop('Label', axis=1)
# y = df['Label']

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y  # ← critical
# )
# X_train.shape, X_test.shape
# X_sample = X_train.sample(10000, random_state=42)
# y_sample = y_train.loc[X_sample.index]
# X_sample.shape


((277498, 66), (69375, 66))

In [3]:
sample_df, _ = train_test_split(df, train_size=7000, stratify=df['Label'], random_state=42)
X = sample_df.drop('Label', axis=1)
y = sample_df['Label']

X_train, X_test , y_train, y_test = train_test_split(X, y, test_size=2000, stratify=y, random_state=42);
X.shape, X_train.shape, X_test.shape

((7000, 66), (5000, 66), (2000, 66))

#### RandomizedSearchCV


In [17]:
param_dist = {
    
    "n_estimators": [ 100, 150, 200, 250, 300, 350],
    
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    
    "max_depth": [3, 4, 5, 6, 7, 8, 9, 10],
    
    "min_samples_split": [2, 5, 10],
    
    "min_samples_leaf": [1, 2, 4],
    
    "subsample": [0.6, 0.8, 1.0]
}

random_search = RandomizedSearchCV(

    estimator=GradientBoostingClassifier(random_state=42),

    param_distributions=param_dist,

    n_iter=30,

    scoring="accuracy",

    cv=StratifiedKFold(n_splits=5),

    verbose=2,

    random_state=42,

    n_jobs=-1
)

In [18]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,GradientBoost...ndom_state=42)
,param_distributions,"{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,30
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo...shuffle=False)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [19]:
print("Best Parameters:", random_search.best_params_)

Best Parameters: {'subsample': 0.8, 'n_estimators': 350, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 5, 'learning_rate': 0.01}


In [ ]:
import joblib

joblib.dump(random_search, "../models/ddos_gd_rs_hyperparter.pkl")

- best model prediction

In [11]:
# best_random_model = random_search.best_estimator_
random_gd = joblib.load('../models/ddos_gd_rs_hyperparter.pkl')

best_random_model = random_gd.best_estimator_

y_pred_random = best_random_model.predict(X_test)

In [12]:
print("Random Search Accuracy:", accuracy_score(y_test, y_pred_random))

print(classification_report(y_test, y_pred_random))

Random Search Accuracy: 0.809
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       173
           1       0.77      0.53      0.63       173
           2       0.62      0.71      0.66       173
           3       0.91      0.95      0.93       173
           4       0.99      1.00      1.00       173
           5       0.76      0.53      0.62       173
           6       0.64      0.87      0.74       173
           7       0.72      0.82      0.77       173
           8       0.72      0.76      0.74       173
           9       0.99      1.00      1.00       173
          10       0.75      0.68      0.72       173
          11       0.99      0.92      0.95        97

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000



#### BayesSearchCV

In [6]:
search_space = {

    "n_estimators": Integer(100, 400),

    "learning_rate": Real(0.01, 0.2),

    "max_depth": Integer(3, 10),

    "min_samples_split": Integer(2, 10),

    "min_samples_leaf": Integer(1, 4),

    "subsample": Real(0.6, 1.0)
}

bayes_search = BayesSearchCV(

    estimator=GradientBoostingClassifier(random_state=42),

    search_spaces=search_space,

    n_iter=30,

    scoring="accuracy",

    cv=StratifiedKFold(n_splits=5),

    verbose=3,

    random_state=42,

    n_jobs=-1
)

In [7]:
bayes_search.fit(X_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

,estimator,GradientBoost...ndom_state=42)
,search_spaces,"{'learning_rate': Real(low=0.01...m='normalize'), 'max_depth': Integer(low=3...m='normalize'), 'min_samples_leaf': Integer(low=1...m='normalize'), 'min_samples_split': Integer(low=2...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,30
,scoring,'accuracy'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,StratifiedKFo...shuffle=False)


In [8]:
print("Best Parameters:", bayes_search.best_params_)

Best Parameters: OrderedDict({'learning_rate': 0.026787675725273456, 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 3, 'n_estimators': 306, 'subsample': 0.8892102570849953})


In [9]:
best_bayes_model = bayes_search.best_estimator_

y_pred_bayes = best_bayes_model.predict(X_test)

#### Compare Bothe Models

In [13]:
random_acc = accuracy_score(y_test, y_pred_random)

bayes_acc = accuracy_score(y_test, y_pred_bayes)

print("Random Search Accuracy:", random_acc)
print("Bayesian Search Accuracy:", bayes_acc)

Random Search Accuracy: 0.809
Bayesian Search Accuracy: 0.751


In [14]:
if bayes_acc > random_acc:
    best_model = best_bayes_model
    print("Bayesian model selected")
else:
    best_model = best_random_model
    print("Random Search model selected")

Random Search model selected


#### Save model

In [15]:
import joblib

joblib.dump(best_model, "../models/ddos_gradient_boost_model.pkl")

['../models/ddos_gradient_boost_model.pkl']